# CTC Sequence Model Training & Evaluation (`asl_ctc_v6.pth`)

This notebook covers the temporal sequence architecture (`SeqNet` in `src/ctc_model.py`), training pipeline (`scripts/train_ctc.py`), checkpoint inspection for `trained_models/ctc/asl_ctc_v6.pth`, and honest evaluation on real continuous word recordings (`data/words_test`).

## How `SeqNet` + CTC Works
1. **Input Representation:** Hand landmarks + motion/presence features (`FEAT_DIM`) resampled over a `4.0s` sliding window (`T` frames).
2. **Temporal Backbone (`SeqNet`):** 1D convolutional feature extractor + bidirectional recurrent layers projecting frame-level logits over `VOCAB` (including CTC blank and `SPACE`).
3. **Training Strategy (`scripts/train_ctc.py`):**
   * Dynamically stitches isolated letter takes from `data/takes` into multi-letter words every epoch so the model never sees identical timing twice.
   * Mixes real continuous words from `data/words_train` (`--words-dir`) while excluding `data/words_test` (`--exclude-words-dir`) to prevent word-level data leakage.
4. **Streaming Inference (`src/stream_decoder.py`):** Decodes overlapping windows in chronological order without future lookahead.

In [ ]:
from pathlib import Path
import sys

# Resolve project root whether launched from repo root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src path     : {SRC_DIR} (exists={SRC_DIR.exists()})")

## 1. Inspect `trained_models/ctc/asl_ctc_v6.pth` Checkpoint
Verify the saved checkpoint dictionary keys (`model_state`, `vocab`, `config`), model parameter count, and layer structure.

In [ ]:
import torch
from ctc_model import load_checkpoint

ckpt_path = PROJECT_ROOT / "trained_models" / "ctc" / "asl_ctc_v6.pth"
raw_ckpt = torch.load(ckpt_path, map_location="cpu")

print("Checkpoint keys :", list(raw_ckpt.keys()))
print("Saved config    :", raw_ckpt.get("config"))
print("Saved vocab     :", raw_ckpt.get("vocab"))

ctc_model, vocab = load_checkpoint(str(ckpt_path))
ctc_model.eval()

total_params = sum(p.numel() for p in ctc_model.parameters())
trainable_params = sum(p.numel() for p in ctc_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,} (trainable: {trainable_params:,})")
print("\nModel Architecture:\n", ctc_model)

## 2. Evaluate `asl_ctc_v6.pth` on Real Recorded Words (`data/words_test`)
Run `scripts/eval_words.py --old ""` so the evaluation scores `asl_ctc_v6.pth` directly across all 91 held-out takes (`leyu_test`) and prints per-word exact match counts and error breakdowns.

In [ ]:
import subprocess

cmd = [
    sys.executable,
    str(SCRIPTS_DIR / "eval_words.py"),
    "--words-dir", str(PROJECT_ROOT / "data" / "words_test"),
    "--ctc", str(ckpt_path),
    "--old", "",
    "--verbose",
]
res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:\n", res.stderr)

## 3. Reference Training & Fine-Tuning Commands
To reproduce training or fine-tune from `asl_ctc_v6.pth` into a new checkpoint (without overwriting `v6`), run from the repository root:

```bash
# Train from scratch with word exclusion
python scripts/train_ctc.py --data data/takes --test-dir data/test --words-dir data/words_train --exclude-words-dir data/words_test --out trained_models/ctc/asl_ctc_v7.pth

# Fine-tune starting from v6 with a lower learning rate
python scripts/train_ctc.py --init trained_models/ctc/asl_ctc_v6.pth --out trained_models/ctc/asl_ctc_v6_finetuned.pth --words-dir data/words_train --exclude-words-dir data/words_test --lr 0.0003 --epochs 20
```